#Setup

In [1]:
# Use legacy tf.keras loader so old .h5 models deserialize correctly
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd

print("TF:", tf.__version__)
print("keras path:", keras.__file__)


TF: 2.19.0
keras path: /usr/local/lib/python3.12/dist-packages/tf_keras/api/_v2/keras/__init__.py


#Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# ---- EDIT THIS PATH ----
PROJECT_DIR = "/content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction"

MODEL_PATH  = f"{PROJECT_DIR}/1DCNN_FeatureExtractor_32.h5"
DATA_CSV    = f"{PROJECT_DIR}/filtered_dataset.csv"    # must have age, sex, label
NPZ_PATH    = f"{PROJECT_DIR}/ecg_dataset.npz"         # saved from your preprocessing
OUT_CSV     = f"{PROJECT_DIR}/features_with_metadata_dense32.csv"

# Load NPZ (your file has keys 'x' and 'y')
with np.load(NPZ_PATH) as f:
    print("Keys in NPZ:", f.files)     # should print ['x','y']
    X = f["x"]                         # signals -> (N, 5000, 12)
    y = f["y"] if "y" in f.files else None

print("Raw X shape:", X.shape)
print("y available:", y is not None)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Keys in NPZ: ['x', 'y']
Raw X shape: (15075, 5000, 12)
y available: True


#Custom metric

In [3]:
from tensorflow.keras.metrics import Metric

class F1ScoreBinary(Metric):
    def __init__(self, name="f1", threshold=0.5, **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.tp = self.add_weight(name="tp", initializer="zeros")
        self.fp = self.add_weight(name="fp", initializer="zeros")
        self.fn = self.add_weight(name="fn", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        import tensorflow as tf
        y_pred = tf.cast(y_pred >= self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        tp = tf.reduce_sum(y_true * y_pred)
        fp = tf.reduce_sum((1 - y_true) * y_pred)
        fn = tf.reduce_sum(y_true * (1 - y_pred))
        self.tp.assign_add(tp); self.fp.assign_add(fp); self.fn.assign_add(fn)

    def result(self):
        return 2 * self.tp / (2 * self.tp + self.fp + self.fn + 1e-8)

    def reset_state(self):
        for v in (self.tp, self.fp, self.fn):
            v.assign(0.0)


#Load the feature model

In [4]:
from tensorflow.keras.models import load_model, Model

# 1) Load the full model (legacy loader already set in Cell 1)
custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
base = load_model(MODEL_PATH, compile=False, custom_objects=custom_objs)
print("Full model input :", base.inputs[0].shape)   # (None, 5000, 12)
print("Full model output:", base.outputs[0].shape)  # (None, 1)

# 2) Get the nested backbone that outputs 32 dims
backbone = base.get_layer("sequential")   # this is the (None, 32) block in your printout

# 3) IMPORTANT: call the backbone on the original input to keep the graph connected
feat_tensor = backbone(base.input)        # <— produces a connected (None, 32) tensor
feature_model = Model(inputs=base.input, outputs=feat_tensor)

print("Feature model output:", feature_model.output_shape)  # expect (None, 32)
print("X shape used:", X.shape)

# 4) Extract features
features = feature_model.predict(X, batch_size=128, verbose=1)
print("Features shape:", features.shape)  # expect (N, 32)


Full model input : (None, 5000, 12)
Full model output: (None, 1)
Feature model output: (None, 32)
X shape used: (15075, 5000, 12)
118/118 [==============================] - 3s 17ms/step
Features shape: (15075, 32)


#Extract features → merge metadata → save CSV

In [5]:
import pandas as pd
import numpy as np

# Re-read metadata
meta_full = pd.read_csv(DATA_CSV)

# Make sure the label column exists and is numeric 0/1
rename_map = {"Age":"age","AGE":"age","Sex":"sex","Gender":"sex","gender":"sex",
              "Label":"label","target":"label","y":"label"}
meta_full = meta_full.rename(columns={k:v for k,v in rename_map.items() if k in meta_full.columns})
assert "label" in meta_full.columns, "Your CSV must contain a label/Label/target column."

csv_labels = pd.to_numeric(meta_full["label"], errors="coerce").fillna(-1).astype(int).to_numpy()
N = len(y)   # should be 15075
print("CSV rows:", len(meta_full), "| NPZ y len:", N)


CSV rows: 15107 | NPZ y len: 15075


In [6]:
# ------- Try A) monotonic subsequence alignment -------
sel_idx = []
i = 0
for j, lbl in enumerate(csv_labels):
    if i == N:
        break
    if lbl == int(y[i]):
        sel_idx.append(j)
        i += 1

print(f"[Subsequence] matched {i}/{N}")
ok = (i == N)

# Optional sanity: compute agreement if subsequence succeeded
if ok:
    meta_aligned = meta_full.iloc[sel_idx].reset_index(drop=True)
    agree = (pd.to_numeric(meta_aligned["label"], errors="coerce").fillna(-1).astype(int).to_numpy() == y).mean()
    print(f"[Subsequence] agreement: {agree:.4f}")

# ------- Try B) chunked sliding if needed -------
if not ok:
    print("[Subsequence] failed; trying chunked sliding…")
    chunk = 200           # tune if needed (100–500)
    sel_idx = []
    start_csv = 0
    ok = True
    for start_y in range(0, N, chunk):
        end_y = min(N, start_y + chunk)
        window_y = y[start_y:end_y]
        best_j, best_score = None, -1.0
        search_limit = min(len(csv_labels), start_csv + 8000)

        for j in range(start_csv, search_limit - (end_y - start_y) + 1):
            score = (csv_labels[j:j + (end_y - start_y)] == window_y).mean()
            if score > best_score:
                best_score, best_j = score, j
            if best_score == 1.0:
                break

        if best_j is None:
            ok = False
            break

        sel_idx.extend(range(best_j, best_j + (end_y - start_y)))
        start_csv = best_j + (end_y - start_y)

    if ok and len(sel_idx) == N:
        meta_aligned = meta_full.iloc[sel_idx].reset_index(drop=True)
        agree = (pd.to_numeric(meta_aligned["label"], errors="coerce").fillna(-1).astype(int).to_numpy() == y).mean()
        print(f"[Chunked] agreement: {agree:.4f}")
    else:
        ok = False

# ------- Final check -------
if not ok:
    raise ValueError(
        "Could not confidently align CSV to NPZ using labels alone.\n"
        "Since we can’t edit preprocessing to add an ID, there’s no safe way to align."
    )

# Keep useful metadata columns
keep_cols = [c for c in ["age","sex","label","ecg_id","patient_id","filename_lr","filename_hr"] if c in meta_aligned.columns]
meta_aligned = meta_aligned[keep_cols].reset_index(drop=True)
print("Aligned metadata shape:", meta_aligned.shape)


[Subsequence] matched 15075/15075
[Subsequence] agreement: 1.0000
Aligned metadata shape: (15075, 7)


#Normal

In [22]:
# === Build final DataFrame and save ===

# 1) Features -> DataFrame
feat_cols = [f"feature_{i}" for i in range(32)]
df_feat = pd.DataFrame(features, columns=feat_cols)

# 2) Concatenate with aligned metadata
out_df = pd.concat([df_feat, meta_aligned], axis=1)

# 3) Drop unwanted ID/filename columns if they exist
drop_cols = ["ecg_id", "patient_id", "filename_lr", "filename_hr"]
out_df = out_df.drop(columns=[c for c in drop_cols if c in out_df.columns])

# 4) Save cleaned CSV
OUT_CSV = f"{PROJECT_DIR}/features_with_metadata_dense32.csv"
out_df.to_csv(OUT_CSV, index=False)

# 5) Quick sanity check
print("Saved cleaned ->", OUT_CSV)
print("Final shape:", out_df.shape)
print("Columns preview:", out_df.columns.tolist()[:8], "...", out_df.columns.tolist()[-3:])
display(out_df.head(3))


Saved cleaned -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/features_with_metadata_dense32.csv
Final shape: (15075, 35)
Columns preview: ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7'] ... ['age', 'sex', 'label']


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,age,sex,label
0,0.251021,1.495924,1.834404,1.597841,0.057906,0.145051,0.0,0.0,0.000000,0.290047,...,0.358492,0.802711,1.214853,0.0,0.121276,0.0,0.542367,56.0,1,0
1,0.000000,1.429176,1.701894,1.408363,0.099227,0.077582,0.0,0.0,0.005201,0.199819,...,0.385609,0.929934,1.049382,0.0,0.136165,0.0,0.779875,19.0,0,0
2,0.074323,1.344176,1.667296,1.423092,0.055575,0.206953,0.0,0.0,0.148224,0.223277,...,0.269637,0.905394,1.104064,0.0,0.088657,0.0,0.596016,37.0,1,0


In [7]:
# --- Utilities for building feature models and saving ---
from tensorflow.keras.models import load_model, Model
import pandas as pd

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
BACKBONE_NAME = "sequential"   # change if your backbone is named differently
BATCH_SIZE = 128               # lower if you get OOM errors

def build_feature_model(h5_path, backbone_name=BACKBONE_NAME, custom_objects=custom_objs):
    base = load_model(h5_path, compile=False, custom_objects=custom_objects)
    if isinstance(base.output_shape, tuple) and base.output_shape[-1] in (32,64,128,256):
        return base
    try:
        backbone = base.get_layer(backbone_name)
        return Model(inputs=base.input, outputs=backbone(base.input))
    except:
        pass
    last_layer = base.layers[-1]
    pre = last_layer.input
    if isinstance(pre, (list, tuple)): pre = pre[0]
    try:
        producer = pre._keras_history.layer
    except AttributeError:
        producer = pre._keras_history[0]
    return Model(inputs=base.input, outputs=producer.output)

def extract_and_save(feature_model, X, meta_aligned, out_csv, drop_extra=True):
    feats = feature_model.predict(X, batch_size=BATCH_SIZE, verbose=1)
    df_feat = pd.DataFrame(feats, columns=[f"feature_{i}" for i in range(feats.shape[1])])
    keep = [c for c in ["age","sex","label","ecg_id","patient_id","filename_lr","filename_hr"] if c in meta_aligned.columns]
    out_df = pd.concat([df_feat, meta_aligned[keep].reset_index(drop=True)], axis=1)
    if drop_extra:
        out_df = out_df.drop(columns=[c for c in ["ecg_id","patient_id","filename_lr","filename_hr"] if c in out_df.columns])
    out_df.to_csv(out_csv, index=False)
    print(f"Saved -> {out_csv} | shape: {out_df.shape}")
    return out_df


In [13]:
# 1) Clear TF state (good hygiene before rebuilding)
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()

from tensorflow.keras.models import load_model, Model
# --- 64-dim (uses the nested backbone 'sequential_2') ---
MODEL64 = f"{PROJECT_DIR}/1DCNN_FeatureExtractor_64.h5"

# If you're using the helper:
fm64 = build_feature_model(MODEL64, backbone_name="sequential_2",
                           custom_objects={"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()})

# If you prefer explicit (equivalent to the helper for this case):
# base64 = load_model(MODEL64, compile=False, custom_objects={"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()})
# fm64 = Model(inputs=base64.input, outputs=base64.get_layer("sequential_2")(base64.input))

print("Feature model (64) output:", fm64.output_shape)

csv64 = f"{PROJECT_DIR}/features_with_metadata_dense64.csv"
df64  = extract_and_save(fm64, X, meta_aligned, csv64, drop_extra=True)


Feature model (64) output: (None, 64)
118/118 [==============================] - 5s 37ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/features_with_metadata_dense64.csv | shape: (15075, 67)


In [10]:
# 1) Clear TF state (good hygiene before rebuilding)
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()

# 2) Load the full 128-dim model
from tensorflow.keras.models import load_model, Model

MODEL128 = f"{PROJECT_DIR}/1DCNN_FeatureExtractor_128.h5"
fm128 = build_feature_model(
    MODEL128,
    backbone_name="sequential_4",   # <-- use sequential_4 for 128-dim
    custom_objects={"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
)
print("Feature model (128) output:", fm128.output_shape)

csv128 = f"{PROJECT_DIR}/features_with_metadata_dense128.csv"
df128  = extract_and_save(fm128, X, meta_aligned, csv128, drop_extra=True)



Feature model (128) output: (None, 128)
118/118 [==============================] - 12s 90ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/features_with_metadata_dense128.csv | shape: (15075, 131)


In [12]:
# 1) Clear TF state (good hygiene before rebuilding)
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

# 2) Build the 256-dim feature model
from tensorflow.keras.models import load_model, Model

MODEL256 = f"{PROJECT_DIR}/1DCNN_FeatureExtractor_256.h5"
fm256 = build_feature_model(
    MODEL256,
    backbone_name="sequential_6",   # <-- per your printout for 256
    custom_objects={"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
)
print("Feature model (256) output:", fm256.output_shape)

# 3) Extract + save
csv256 = f"{PROJECT_DIR}/features_with_metadata_dense256.csv"
df256  = extract_and_save(fm256, X, meta_aligned, csv256, drop_extra=True)


Feature model (256) output: (None, 256)
118/118 [==============================] - 32s 235ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/features_with_metadata_dense256.csv | shape: (15075, 259)


In [14]:
# --- Helper: find a nested backbone layer that outputs the target dim ---
from tensorflow.keras.models import load_model, Model

def find_backbone_name(h5_path, custom_objects, target_dim):
    """Return the name of a nested Model/Sequential layer whose output ends with target_dim.
       If none found, return None (the build_feature_model fallback will handle it)."""
    base = load_model(h5_path, compile=False, custom_objects=custom_objects)
    for lyr in reversed(base.layers):
        cls = lyr.__class__.__name__
        if "Model" in cls or "Sequential" in cls:
            try:
                shp = lyr.output_shape
                if isinstance(shp, tuple) and shp[-1] == target_dim:
                    return lyr.name
            except Exception:
                pass
    return None  # fallback path will still work


#Over Sampled

In [17]:
# --- Over_Sampled 32 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Over_Sampled_1DCNN_FeatureExtractor_32 .h5"
bname = find_backbone_name(h5, custom_objs, 32)  # might be None if top-level already outputs 32
print("Backbone candidate (32):", bname)

fm32_os = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (32) output:", fm32_os.output_shape)

csv32_os = f"{PROJECT_DIR}/over_sampled_1d_cnn_features_32.csv"
df32_os = extract_and_save(fm32_os, X, meta_aligned, csv32_os, drop_extra=True)


Backbone candidate (32): sequential
Feature model (32) output: (None, 32)
118/118 [==============================] - 2s 16ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/over_sampled_1d_cnn_features_32.csv | shape: (15075, 35)


In [18]:
# --- Over_Sampled 64 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Over_Sampled_1DCNN_FeatureExtractor_64 .h5"
bname = find_backbone_name(h5, custom_objs, 64)  # often 'sequential_2'
print("Backbone candidate (64):", bname)

fm64_os = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (64) output:", fm64_os.output_shape)

csv64_os = f"{PROJECT_DIR}/over_sampled_1d_cnn_features_64.csv"
df64_os = extract_and_save(fm64_os, X, meta_aligned, csv64_os, drop_extra=True)


Backbone candidate (64): sequential_2
Feature model (64) output: (None, 64)
118/118 [==============================] - 4s 35ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/over_sampled_1d_cnn_features_64.csv | shape: (15075, 67)


In [19]:
# --- Over_Sampled 128 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Over_Sampled_1DCNN_FeatureExtractor_128 .h5"
bname = find_backbone_name(h5, custom_objs, 128)  # often 'sequential_4'
print("Backbone candidate (128):", bname)

fm128_os = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (128) output:", fm128_os.output_shape)

csv128_os = f"{PROJECT_DIR}/over_sampled_1d_cnn_features_128.csv"
df128_os = extract_and_save(fm128_os, X, meta_aligned, csv128_os, drop_extra=True)


Backbone candidate (128): sequential_4
Feature model (128) output: (None, 128)
118/118 [==============================] - 10s 81ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/over_sampled_1d_cnn_features_128.csv | shape: (15075, 131)


In [20]:
# --- Over_Sampled 256 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Over_Sampled_1DCNN_FeatureExtractor_256.h5"
bname = find_backbone_name(h5, custom_objs, 256)  # often 'sequential_6'
print("Backbone candidate (256):", bname)

fm256_os = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (256) output:", fm256_os.output_shape)

csv256_os = f"{PROJECT_DIR}/over_sampled_1d_cnn_features_256.csv"
df256_os = extract_and_save(fm256_os, X, meta_aligned, csv256_os, drop_extra=True)


Backbone candidate (256): sequential_6
Feature model (256) output: (None, 256)
118/118 [==============================] - 24s 205ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/over_sampled_1d_cnn_features_256.csv | shape: (15075, 259)


#Under Sampled

In [21]:
# --- Under_Sampled 32 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Under_Sampled_1DCNN_FeatureExtractor_32.h5"
bname = find_backbone_name(h5, custom_objs, 32)   # may be None if top-level outputs 32
print("Backbone candidate (32):", bname)

fm32_us = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (32) output:", fm32_us.output_shape)

csv32_us = f"{PROJECT_DIR}/under_sampled_1d_cnn_features_32.csv"
df32_us  = extract_and_save(fm32_us, X, meta_aligned, csv32_us, drop_extra=True)


Backbone candidate (32): sequential
Feature model (32) output: (None, 32)
118/118 [==============================] - 2s 16ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/under_sampled_1d_cnn_features_32.csv | shape: (15075, 35)


In [22]:
# --- Under_Sampled 64 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Under_Sampled_1DCNN_FeatureExtractor_64.h5"
bname = find_backbone_name(h5, custom_objs, 64)   # often 'sequential_2'
print("Backbone candidate (64):", bname)

fm64_us = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (64) output:", fm64_us.output_shape)

csv64_us = f"{PROJECT_DIR}/under_sampled_1d_cnn_features_64.csv"
df64_us  = extract_and_save(fm64_us, X, meta_aligned, csv64_us, drop_extra=True)


Backbone candidate (64): sequential_2
Feature model (64) output: (None, 64)
118/118 [==============================] - 4s 35ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/under_sampled_1d_cnn_features_64.csv | shape: (15075, 67)


In [23]:
# --- Under_Sampled 128 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Under_Sampled_1DCNN_FeatureExtractor_128.h5"
bname = find_backbone_name(h5, custom_objs, 128)  # often 'sequential_4'
print("Backbone candidate (128):", bname)

fm128_us = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (128) output:", fm128_us.output_shape)

csv128_us = f"{PROJECT_DIR}/under_sampled_1d_cnn_features_128.csv"
df128_us  = extract_and_save(fm128_us, X, meta_aligned, csv128_us, drop_extra=True)


Backbone candidate (128): sequential_4
Feature model (128) output: (None, 128)
118/118 [==============================] - 10s 81ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/under_sampled_1d_cnn_features_128.csv | shape: (15075, 131)


In [24]:
# --- Under_Sampled 256 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Under_Sampled_1DCNN_FeatureExtractor_256.h5"
bname = find_backbone_name(h5, custom_objs, 256)  # often 'sequential_6'
print("Backbone candidate (256):", bname)

fm256_us = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (256) output:", fm256_us.output_shape)

csv256_us = f"{PROJECT_DIR}/under_sampled_1d_cnn_features_256.csv"
df256_us  = extract_and_save(fm256_us, X, meta_aligned, csv256_us, drop_extra=True)


Backbone candidate (256): sequential_6
Feature model (256) output: (None, 256)
118/118 [==============================] - 24s 207ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/under_sampled_1d_cnn_features_256.csv | shape: (15075, 259)


#Weighted

In [25]:
# --- Weighted 32 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Weighted_1DCNN_FeatureExtractor_32.h5"
bname = find_backbone_name(h5, custom_objs, 32)  # may be None if top-level already outputs 32
print("Backbone candidate (32):", bname)

fm32_w = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (32) output:", fm32_w.output_shape)

csv32_w = f"{PROJECT_DIR}/weighted_1d_cnn_features_32.csv"
df32_w  = extract_and_save(fm32_w, X, meta_aligned, csv32_w, drop_extra=True)


Backbone candidate (32): sequential
Feature model (32) output: (None, 32)
118/118 [==============================] - 2s 16ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/weighted_1d_cnn_features_32.csv | shape: (15075, 35)


In [26]:
# --- Weighted 64 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Weighted_1DCNN_FeatureExtractor_64.h5"
bname = find_backbone_name(h5, custom_objs, 64)  # often 'sequential_2'
print("Backbone candidate (64):", bname)

fm64_w = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (64) output:", fm64_w.output_shape)

csv64_w = f"{PROJECT_DIR}/weighted_1d_cnn_features_64.csv"
df64_w  = extract_and_save(fm64_w, X, meta_aligned, csv64_w, drop_extra=True)


Backbone candidate (64): sequential_2
Feature model (64) output: (None, 64)
118/118 [==============================] - 4s 35ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/weighted_1d_cnn_features_64.csv | shape: (15075, 67)


In [27]:
# --- Weighted 128 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Weighted_1DCNN_FeatureExtractor_128.h5"
bname = find_backbone_name(h5, custom_objs, 128)  # often 'sequential_4'
print("Backbone candidate (128):", bname)

fm128_w = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (128) output:", fm128_w.output_shape)

csv128_w = f"{PROJECT_DIR}/weighted_1d_cnn_features_128.csv"
df128_w  = extract_and_save(fm128_w, X, meta_aligned, csv128_w, drop_extra=True)


Backbone candidate (128): sequential_4
Feature model (128) output: (None, 128)
118/118 [==============================] - 10s 81ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/weighted_1d_cnn_features_128.csv | shape: (15075, 131)


In [28]:
# --- Weighted 256 ---
import tensorflow as tf, gc
tf.keras.backend.clear_session(); gc.collect()
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

custom_objs = {"F1ScoreBinary": F1ScoreBinary, "f1": F1ScoreBinary()}
h5 = f"{PROJECT_DIR}/Weighted_1DCNN_FeatureExtractor_256.h5"
bname = find_backbone_name(h5, custom_objs, 256)  # often 'sequential_6'
print("Backbone candidate (256):", bname)

fm256_w = build_feature_model(h5, backbone_name=bname or "sequential", custom_objects=custom_objs)
print("Feature model (256) output:", fm256_w.output_shape)

csv256_w = f"{PROJECT_DIR}/weighted_1d_cnn_features_256.csv"
df256_w  = extract_and_save(fm256_w, X, meta_aligned, csv256_w, drop_extra=True)


Backbone candidate (256): sequential_6
Feature model (256) output: (None, 256)
118/118 [==============================] - 24s 206ms/step
Saved -> /content/drive/MyDrive/Second year Research/1dCNNFeatureExtraction/weighted_1d_cnn_features_256.csv | shape: (15075, 259)
